<a href="https://colab.research.google.com/github/shubhamparmarp70/AlexNet-Model/blob/main/AlexNet_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import Libraries
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [ ]:
# Data Preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),   # helps your dataset
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

In [ ]:
# Load Dataset
from google.colab import drive
drive.mount('/content/drive')

dataset_path = "/content/drive/MyDrive/dataset"

train_data = datasets.ImageFolder(dataset_path + "/train", transform=transform)
test_data = datasets.ImageFolder(dataset_path + "/test", transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print("Classes:", train_data.classes)

Mounted at /content/drive
Classes: ['Augmented-images', 'LNS Occluded', 'covering', 'glasses', 'plain', 'sunglasses', 'sunglasses-imagenet', 'unAugmented-images']


In [ ]:
# Load “Modern AlexNet”
model = models.alexnet(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:06<00:00, 39.1MB/s]


In [ ]:
# Modify for Your Dataset
num_classes = len(train_data.classes)

model.classifier[6] = nn.Linear(4096, num_classes)

In [ ]:
# Move to Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
# Training Setup
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
# Training Loop (with Accuracy)
epochs = 10

for epoch in range(epochs):
    model.train()

    correct = 0
    total = 0
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    print(f"Epoch {epoch+1}")
    print(f"Loss: {running_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")

Epoch 1
Loss: 129.4079
Train Accuracy: 87.96%
Epoch 2
Loss: 77.9980
Train Accuracy: 92.02%
Epoch 3
Loss: 64.7860
Train Accuracy: 93.66%
Epoch 4
Loss: 56.0179
Train Accuracy: 94.59%
Epoch 5
Loss: 51.7914
Train Accuracy: 95.01%
Epoch 6
Loss: 45.0011
Train Accuracy: 95.71%
Epoch 7
Loss: 40.3872
Train Accuracy: 95.96%
Epoch 8
Loss: 35.5706
Train Accuracy: 96.54%
Epoch 9
Loss: 31.6133
Train Accuracy: 96.99%
Epoch 10
Loss: 34.5722
Train Accuracy: 96.97%


In [ ]:
# Test Accuracy
correct = 0
total = 0

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = 100 * correct / total
print(f"✅ Test Accuracy: {test_acc:.2f}%")

✅ Test Accuracy: 94.91%
